<a href="https://colab.research.google.com/github/vighi2004/Interactive_TextBook/blob/main/InteractiveTextBook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!pip install \
langchain==0.3.27 \
langchain-core==0.3.76 \
langchain-community==0.3.29 \
langchain-openai==0.3.31 \
langchain-chroma==0.2.6 \
rank_bm25 \
ragas==0.4.3 \
chromadb \
pypdf

  Using cached langchain_core-0.3.76-py3-none-any.whl.metadata (3.7 kB)
  Using cached langchain_openai-0.3.31-py3-none-any.whl.metadata (2.4 kB)
  Using cached openai-1.109.1-py3-none-any.whl.metadata (29 kB)
INFO: pip is looking at multiple versions of instructor to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of instructor to determine which version is compatible with other requirements. This could take a while.
Using cached langchain_core-0.3.76-py3-none-any.whl (447 kB)
Using cached langchain_openai-0.3.31-py3-none-any.whl (74 kB)
Using cached openai-1.109.1-py3-none-any.whl (948 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.9/157.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.0/352.0 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.2/226.2 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 9.6 MB/s eta 0

STEP1:Intialization of phase creating LLM and Embedding model

In [21]:
from google.colab import userdata
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

#LLM
llm = ChatOpenAI(
    model="google/gemini-2.5-flash",
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0,
    max_tokens=2048
)
#Embedding model
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1"
)
print("✅ LLM and Embedding Models initialized successfully.")

✅ LLM and Embedding Models initialized successfully.


In [2]:
response=llm.invoke("hello")
print(response.content);

Hello! How can I help you today?


STEP 2:Documnet uploadings all which are saved in file directory textBooks.

In [3]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader

# Load all PDF files from the textbooks folder
loader = PyPDFDirectoryLoader("textBooks/")
raw_documents = loader.load()

for doc in raw_documents:
    file_name = os.path.basename(doc.metadata.get("source", "Unknown"))
    page_num = doc.metadata.get("page", 0) + 1

    doc.metadata["file_name"] = file_name
    doc.metadata["page_number"] = page_num

print(f"✅ Loaded {len(raw_documents)} total pages across all textbooks.")


✅ Loaded 138 total pages across all textbooks.


STEP 3:Chunkings with help of langchain method that is recursiveSpiltter


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(raw_documents)

print(f"✅ Created {len(chunks)} text chunks from the loaded textbooks.")

✅ Created 391 text chunks from the loaded textbooks.


STEP 4:Combine Chroma Vector Store  with BM25  inside an EnsembleRetriever for hybrid retrieval.

In [16]:
import os
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.ensemble import EnsembleRetriever

PERSIST_DIR = "./Vector_DB"

# 1. Dense Semantic Vector Search via Chroma
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR
)
vector_retriever = vector_db.as_retriever(search_kwargs={"k": 2})

# 2. Sparse Keyword Search via BM25
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 2

# 3. Hybrid Search: Ensemble Retriever (50% Semantic + 50% Keyword)
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5]
)

print("✅ Persistent Chroma DB created and Hybrid Search (BM25 + Vector) initialized.")

✅ Persistent Chroma DB created and Hybrid Search (BM25 + Vector) initialized.


STE 6:Chaining process doing with student friendly prompt

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# Prompt for each retrieved document
document_prompt = ChatPromptTemplate.from_template("""
Source:
File: {file_name}
Page: {page_number}

Content:
{page_content}
""")

# Main RAG prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an expert textbook tutor.

Answer ONLY using the retrieved context.

If the answer is not present in the context, say that it is not available.

Retrieved Context:
{context}
"""),
    ("human", "{input}")
])

combine_docs_chain = create_stuff_documents_chain(
    llm,
    prompt,
    document_prompt=document_prompt,
    document_variable_name="context"
)

rag_chain = create_retrieval_chain(
    hybrid_retriever,
    combine_docs_chain
)

STEP 7:asking question and invoking rag_chain

In [9]:
def ask_textbook(question: str):
    print(f"\n🔍 Question: {question}")
    print("=" * 60)

    response = rag_chain.invoke({"input": question})

    print("\n🎓 Answer:")
    print(response["answer"])

    print("\n" + "-" * 60)
    print("📚 Retrieved Context Sources:")
    for i, doc in enumerate(response["context"], 1):
        file = doc.metadata.get("file_name", "N/A")
        page = doc.metadata.get("page_number", "N/A")
        print(f"  [{i}] File: {file} | Page: {page}")

# --- Test Queries ---
ask_textbook("Explain Newton's first law of motion with an example.")
ask_textbook("What is photosynthesis and how does it work?")


🔍 Question: Explain Newton's first law of motion with an example.

🎓 Answer:
The provided text does not contain information about Newton's first law of motion.

------------------------------------------------------------
📚 Retrieved Context Sources:
  [1] File: class7ScienceTextbook.pdf | Page: 108
  [2] File: class7ScienceTextbook.pdf | Page: 61
  [3] File: class7ScienceTextbook.pdf | Page: 119

🔍 Question: What is photosynthesis and how does it work?

🎓 Answer:
Photosynthesis is the process by which green plants synthesize food for themselves. It occurs in the presence of sunlight, using carbon dioxide and water. Chlorophyll, sunlight, carbon dioxide, and water are necessary for this process. During photosynthesis, solar energy is captured by the leaves and stored in the plant as food. Oxygen is also produced and released during photosynthesis.

------------------------------------------------------------
📚 Retrieved Context Sources:
  [1] File: class7ScienceTextbook.pdf | Page: 18

STEP 8;Evlaution of RAG

In [36]:
!pip install -U ragas datasets rapidfuzz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_openai-1.4.1-py3-none-any.whl.metadata (3.4 kB)
INFO: pip is still looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Using cached langchain_core-0.3.86-py3-none-any.whl.metadata (3.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 104.

In [22]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    ContextPrecision,
    ContextRecall,
)

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Wrap your models
evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)


def evaluate_rag(question: str, reference: str):

    # Run your Hybrid RAG
    response = rag_chain.invoke({"input": question})

    answer = response["answer"]
    contexts = [doc.page_content for doc in response["context"]]

    # Dataset for Ragas
    dataset = Dataset.from_dict({
        "question": [question],
        "answer": [answer],
        "contexts": [contexts],
        "reference": [reference]
    })

    # Evaluate
    results = evaluate(
        dataset,
        metrics=[
            Faithfulness(),
            ResponseRelevancy(),
            ContextPrecision(),
            ContextRecall(),
        ],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings
    )

    scores = results.to_pandas().iloc[0]

    print("\n" + "="*80)
    print("🔍 Question:")
    print(question)

    print("\n🎓 Generated Answer:")
    print(answer)

    print("\n📖 Ground Truth:")
    print(reference)

    print("\n📊 Evaluation Scores")
    print(f"Faithfulness      : {scores['faithfulness']:.3f}")
    print(f"Response Relevancy: {scores['answer_relevancy']:.3f}")
    print(f"Context Precision : {scores['context_precision']:.3f}")
    print(f"Context Recall    : {scores['context_recall']:.3f}")

    return scores

/tmp/ipykernel_60059/1792836757.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/tmp/ipykernel_60059/1792836757.py:3: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import (
/tmp/ipykernel_60059/1792836757.py:3: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
/tmp/ipykernel_60059/1792836757.py:3: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be remo

In [15]:
benchmark = [
    {
        "question": "Differentiate between physical and chemical changes.",
        "reference": "A physical change changes only the physical properties such as shape, size, or state of a substance without forming a new substance. A chemical change results in the formation of one or more new substances with different properties and is usually irreversible."
    },
    {
        "question": "What is photosynthesis and how does it occur in plants?",
        "reference": "Photosynthesis is the process by which green plants prepare their own food using carbon dioxide and water in the presence of sunlight and chlorophyll. During this process, glucose is produced as food and oxygen is released into the atmosphere."
    },
    {
        "question": "What are acids and bases? Give one example of each.",
        "reference": "Acids are substances that are sour in taste and turn blue litmus paper red. Bases are substances that are bitter and soapy to touch and turn red litmus paper blue. Example of an acid: Lemon juice. Example of a base: Soap solution."
    }
]

In [26]:
all_results = []

for sample in benchmark:
    score = evaluate_rag(
        sample["question"],
        sample["reference"]
    )
    all_results.append(score)

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)



🔍 Question:
Differentiate between physical and chemical changes.

🎓 Generated Answer:
That's a great question! Understanding the difference between physical and chemical changes is fundamental in science.

Based on what you've learned:

*   **Physical Changes:** These are changes in the **physical properties** of a substance. Think about things like its shape, size, color, or state (like solid to liquid). The key thing here is that **no new substances are formed**. The original substance is still there, just in a different form. These changes can often be **reversible**.

    *   **Example:** Melting of wax. The wax changes from a solid to a liquid, but it's still wax.
    *   **Example:** Dissolving sugar in water. The sugar is still sugar, it's just spread out in the water.

*   **Chemical Changes:** In contrast, chemical changes are all about forming **new substances**. When a chemical change occurs, the original substance is transformed into something entirely different with new p

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]


🔍 Question:
What is photosynthesis and how does it occur in plants?

🎓 Generated Answer:
That's a great question! Let's break down photosynthesis.

Photosynthesis is the process by which plants create their own food. It primarily occurs in the green parts of the plant, especially the leaves, but also in green stems and branches.

Here's how it happens:

*   **Ingredients:** Plants use carbon dioxide from the air and water.
*   **The Special Ingredient:** They have a green pigment called chlorophyll, found in the cells of their leaves.
*   **The Energy Source:** Sunlight provides the energy needed for this process.

During photosynthesis, the chlorophyll-containing cells, in the presence of sunlight, use carbon dioxide and water to produce carbohydrates (which are a type of food for the plant) and release oxygen.

You can think of it like a recipe:

Carbon Dioxide + Water + Sunlight (with Chlorophyll) → Carbohydrates + Oxygen

The presence of starch in leaves is a good indicator that p

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]


🔍 Question:
What are acids and bases? Give one example of each.

🎓 Generated Answer:
That's a great question! Let's break down what acids and bases are using the information provided.

Based on what you've learned:

*   **Acids** are substances that are **sour in taste**. They also have a specific effect on litmus paper: they turn **blue litmus red**.
    *   An example of an acid from the text is **Acetic acid**, which is found in vinegar.

*   **Bases** are substances that are **bitter in taste** and feel **soapy to touch**. They have the opposite effect on litmus paper compared to acids: they turn **red litmus blue**.
    *   An example of a base from the text is **Calcium hydroxide**, which is found in lime water.

It's important to remember the caution mentioned: we should **never taste unknown substances** as they could be harmful!

📖 Ground Truth:
Acids are substances that are sour in taste and turn blue litmus paper red. Bases are substances that are bitter and soapy to touch 

In [27]:
import pandas as pd

df = pd.DataFrame(all_results)

print("\n" + "="*50)
print("📊 Average RAG Evaluation")
print("="*50)

print(f"Faithfulness      : {df['faithfulness'].mean():.3f}")
print(f"Response Relevancy: {df['answer_relevancy'].mean():.3f}")
print(f"Context Precision : {df['context_precision'].mean():.3f}")
print(f"Context Recall    : {df['context_recall'].mean():.3f}")


📊 Average RAG Evaluation
Faithfulness      : 0.923
Response Relevancy: 0.742
Context Precision : 0.380
Context Recall    : 0.833


STEP 9:adding new pdfs wihtout creting seprate pipeline we add in existing vector databse

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

# Add a new PDF to the existing Chroma Vector DB
def add_file(pdf_path: str):

    # Load PDF
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()

    # Add metadata
    file_name = os.path.basename(pdf_path)

    for page in pages:
        page.metadata["file_name"] = file_name
        page.metadata["page_number"] = page.metadata.get("page", 0) + 1

    # Same splitter used while creating DB
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(pages)

    # Add to existing Chroma DB
    vector_db.add_documents(chunks)

    print(f"✅ Added '{file_name}'")
    print(f"📄 Pages : {len(pages)}")
    print(f"🧩 Chunks: {len(chunks)}")
    print(f"📚 Total Chunks in DB: {vector_db._collection.count()}")
add_file("./textbooks/maths7.pdf");

ValueError: File path ./textbooks/maths7.pdf is not a valid file or url

In [25]:
import os

from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.ensemble import EnsembleRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# -----------------------------
# API KEY
# -----------------------------
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

# -----------------------------
# -----------------------------
# Load Existing Chroma DB
# -----------------------------
vector_db = Chroma(
    persist_directory="./Vector_DB",
    embedding_function=embeddings
)

print(f"✅ Existing Chroma DB Loaded ({vector_db._collection.count()} chunks)")

# -----------------------------
# Reload PDFs (Only for BM25)
# -----------------------------
documents = []

for file in os.listdir("./textBooks"):

    if file.endswith(".pdf"):

        loader = PyPDFLoader(os.path.join("./textBooks", file))
        pages = loader.load()

        for page in pages:
            page.metadata["file_name"] = file
            page.metadata["page_number"] = page.metadata.get("page", 0) + 1

        documents.extend(pages)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=150
)

chunks = splitter.split_documents(documents)

print(f"✅ Loaded {len(chunks)} chunks for BM25")

# -----------------------------
# Hybrid Retrieval
# -----------------------------
vector_retriever = vector_db.as_retriever(
    search_kwargs={"k": 4}
)

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 2

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.7, 0.3]
)

# -----------------------------
# Prompt
# -----------------------------
system_prompt = (
    "You are an expert interactive textbook tutor designed to help students learn complex concepts.\n"
    "Answer the student's question clearly, thoroughly, and in an encouraging educational tone using ONLY the context provided.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# -----------------------------
# RAG Chain
# -----------------------------
combine_docs_chain = create_stuff_documents_chain(
    llm,
    prompt
)

rag_chain = create_retrieval_chain(
    hybrid_retriever,
    combine_docs_chain
)

print("✅ Hybrid RAG Ready!")

✅ Existing Chroma DB Loaded (782 chunks)
✅ Loaded 530 chunks for BM25
✅ Hybrid RAG Ready!
